# LatentGate M1 — CogVideoX-5B-I2V T4 smoke test

This notebook confirms the no-paid T4 path before installing packages or project files, installs the pinned package stack, then checks Tesla T4 hardware and package versions before model weights are loaded. It cannot inspect billing class, select a runtime, or override the current quota limit. Do not choose a paid accelerator.

Capture records label phases explicitly: transformer hidden/spatial summaries and predicted-clean x0 come from the current model evaluation at timestep t_i; post_step_i is the scheduler result after that evaluation. For nonfinal steps it becomes the next model input; after the final step it is the terminal latent passed to decoding. The runner also fails closed unless exactly six summary blocks and three spatial-map blocks are selected.

Upload `latentgate_m1_bundle.zip` from the project workspace in the preflight cell. The notebook checks its pinned SHA-256 before extracting the runner, config, fixture description, and fixture images. During the run, it downloads a clearly incomplete baseline-only checkpoint before allowing the exact same-seed replay to start. Colab confirms only that its `files.download` call returned; it cannot confirm that the browser finished saving the file.


In [ ]:
billing_confirmation = input("Before installing packages, confirm Colab Resources says not subscribed and 0 compute units, the selected accelerator is the free T4, and no paid checkout is required. Type FREE T4 to continue: ")
if billing_confirmation.strip() != "FREE T4":
    raise RuntimeError("No-paid T4 was not confirmed; stopping before package installation or project files.")

# Pin the PyTorch/TorchVision pair to the observed CUDA 13.0 T4 stack.
%pip install --index-url https://download.pytorch.org/whl/cu130 --extra-index-url https://pypi.org/simple "torch==2.11.0+cu130" "torchvision==0.26.0+cu130"
%pip install "diffusers==0.40.0" "transformers==5.16.1" "accelerate==1.14.0" "huggingface_hub==1.29.0" "safetensors==0.8.0" "sentencepiece==0.2.2" imageio imageio-ffmpeg PyYAML Pillow
print("Pinned dependencies installed. The next cell verifies the active runtime and refuses unsupported hardware or package versions.")


In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, zipfile
from google.colab import files

billing_confirmation = input("Confirm the connected runtime is still the no-paid Tesla T4 and no paid-resource prompt is present. Type FREE T4 to continue: ")
if billing_confirmation.strip() != "FREE T4":
    raise RuntimeError("Free T4 was not confirmed; stopping before model weights are loaded.")

uploaded = files.upload()
bundle_names = [name for name in uploaded if name.endswith("latentgate_m1_bundle.zip")]
if len(bundle_names) != 1:
    raise RuntimeError("Upload exactly one file named latentgate_m1_bundle.zip")
expected_bundle_sha256 = "7f0764dd3e8730485650d1c80defc4e6024f621af03068dad18291e272daee1e"
actual_bundle_sha256 = hashlib.sha256(uploaded[bundle_names[0]]).hexdigest()
if actual_bundle_sha256 != expected_bundle_sha256:
    raise RuntimeError(f"M1 bundle checksum mismatch: expected {expected_bundle_sha256}, got {actual_bundle_sha256}")

project_root = Path("/content/latentgate")
if project_root.exists():
    raise FileExistsError(f"Refusing to replace existing project directory: {project_root}")
project_root.mkdir(parents=True)
with zipfile.ZipFile(bundle_names[0]) as archive:
    archive.extractall(project_root)
config_path = project_root / "m1/run_config.yaml"
preflight_dir = Path("/content/latentgate_m1_preflight")
subprocess.run([sys.executable, str(project_root / "m1/run_m1.py"), "--config", str(config_path), "--output-dir", str(preflight_dir), "--preflight-only"], cwd=project_root, check=True)
runtime = json.loads((preflight_dir / "runtime_preflight.json").read_text())
config = __import__("yaml").safe_load(config_path.read_text())
if not runtime["cuda_available"] or "T4" not in str(runtime["device_name"]):
    raise RuntimeError(f"A Tesla T4 is required; confirm free billing in the Colab UI; detected {runtime['device_name']!r}. No model weights were loaded.")
if runtime["compute_capability"] != [7, 5]:
    raise RuntimeError(f"Unexpected T4 compute capability: {runtime['compute_capability']}")
mismatches = {name: {"expected": expected, "actual": runtime.get(name)} for name, expected in config["runtime"]["expected_packages"].items() if runtime.get(name) != expected}
if mismatches:
    raise RuntimeError(f"Pinned package mismatch; no model weights were loaded: {mismatches}")
print("Preflight passed:", runtime["device_name"], runtime["compute_capability"], "Python", runtime["python_version"])
print("PyTorch", runtime["torch"], "CUDA", runtime["cuda_runtime"], "Diffusers", runtime["diffusers"])
print("Free GPU memory GiB:", round(runtime["free_memory_bytes"] / 1024**3, 2))

In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys, traceback
from google.colab import files

run_dir = project_root / config["capture"]["output_dir"]
runner_log_path = Path("/content/latentgate_m1_runner.log")
if run_dir.exists():
    raise FileExistsError(f"Refusing to overwrite existing run artifacts: {run_dir}")
if runner_log_path.exists():
    raise FileExistsError(f"Refusing to overwrite existing runner log: {runner_log_path}")

runner_command = [
    sys.executable,
    "-u",
    str(project_root / "m1/run_m1.py"),
    "--config",
    str(config_path),
    "--output-dir",
    str(run_dir),
    "--baseline-checkpoint-handshake",
]
runner_exit_code = None
invocation_error = None
process = None
baseline_checkpoint_ready_seen = False
baseline_checkpoint_ack_sent = False
baseline_checkpoint_acknowledged = False
baseline_checkpoint_saved_confirmation = False
try:
    with runner_log_path.open("w", encoding="utf-8") as runner_log:
        process = subprocess.Popen(
            runner_command,
            cwd=project_root,
            stdin=subprocess.PIPE,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        if process.stdout is None:
            raise RuntimeError("Runner stdout pipe was not created")
        checkpoint_marker = "LATENTGATE_M1_BASELINE_CHECKPOINT_READY="
        for line in process.stdout:
            print(line, end="", flush=True)
            runner_log.write(line)
            runner_log.flush()
            if line.startswith(checkpoint_marker):
                if baseline_checkpoint_ready_seen:
                    raise RuntimeError("Runner emitted the baseline checkpoint marker more than once")
                baseline_checkpoint_ready_seen = True
                checkpoint = json.loads(line[len(checkpoint_marker):])
                checkpoint_path = Path(checkpoint["zip_path"]).resolve()
                expected_checkpoint_path = (
                    run_dir.parent / f"{run_dir.name}_baseline_checkpoint.zip"
                ).resolve()
                if checkpoint_path != expected_checkpoint_path or not checkpoint_path.is_file():
                    raise RuntimeError(
                        "Runner announced an unexpected or missing baseline checkpoint ZIP: "
                        f"{checkpoint_path}"
                    )
                digest_state = hashlib.sha256()
                with checkpoint_path.open("rb") as checkpoint_file:
                    for chunk in iter(lambda: checkpoint_file.read(1024 * 1024), b""):
                        digest_state.update(chunk)
                digest = digest_state.hexdigest()
                if digest != checkpoint.get("sha256"):
                    raise RuntimeError(
                        "Baseline checkpoint ZIP checksum mismatch: "
                        f"runner={checkpoint.get('sha256')}, notebook={digest}"
                    )
                # This initiates the Colab browser download. Its return does not prove that
                # the browser finished saving the ZIP; the runner records that limitation.
                files.download(str(checkpoint_path))
                saved_confirmation = input(
                    f"After checking your local Downloads folder for {checkpoint_path.name}, "
                    "type CHECKPOINT SAVED to let the replay continue: "
                )
                if saved_confirmation.strip() != "CHECKPOINT SAVED":
                    raise RuntimeError(
                        "Baseline checkpoint was not confirmed in local Downloads; replay was stopped."
                    )
                baseline_checkpoint_saved_confirmation = True
                if process.stdin is None:
                    raise RuntimeError("Runner stdin pipe was not created for checkpoint acknowledgment")
                process.stdin.write(f"ACK_BASELINE_CHECKPOINT {digest}\n")
                process.stdin.flush()
                process.stdin.close()
                baseline_checkpoint_ack_sent = True
                print(
                    "The checkpoint appeared in local Downloads per operator confirmation. "
                    "Colab cannot independently verify the browser save; waiting for runner acknowledgment.",
                    flush=True,
                )
            elif line.startswith("LATENTGATE_M1_BASELINE_CHECKPOINT_ACKNOWLEDGED="):
                acknowledgment = json.loads(
                    line.split("=", 1)[1]
                )
                if (
                    not baseline_checkpoint_ack_sent
                    or acknowledgment.get("sha256") != digest
                    or acknowledgment.get("replay_may_start") is not True
                ):
                    raise RuntimeError("Runner returned an invalid baseline checkpoint acknowledgment")
                baseline_checkpoint_acknowledged = True
        runner_exit_code = process.wait()
        if runner_exit_code == 0 and not baseline_checkpoint_acknowledged:
            raise RuntimeError("Runner exited successfully without the required baseline checkpoint handoff")
except BaseException as error:
    if process is not None and process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
    if process is not None:
        runner_exit_code = process.returncode
    invocation_error = {
        "error_type": type(error).__name__,
        "error": str(error),
        "traceback": traceback.format_exc(),
        "baseline_checkpoint_acknowledged": baseline_checkpoint_acknowledged,
        "operator_confirmed_checkpoint_visible_in_downloads": baseline_checkpoint_saved_confirmation,
        "replay_may_have_started": baseline_checkpoint_acknowledged,
        "download_acknowledgment_scope": (
            "Operator reported the ZIP visible in local Downloads after files.download returned; "
            "Colab cannot independently inspect or verify the browser filesystem"
        ),
    }

# Create the output directory only after the runner exits. The runner uses its
# initial existence to decide whether it owns this run and may write failure.json.
run_dir.mkdir(parents=True, exist_ok=True)
if runner_log_path.exists():
    shutil.copy2(runner_log_path, run_dir / "notebook_runner.log")
invocation_record = {
    "status": "completed" if invocation_error is None and runner_exit_code == 0 else "failed",
    "command": runner_command,
    "exit_code": runner_exit_code,
    "error": invocation_error,
    "baseline_checkpoint": {
        "operator_confirmed_visible_in_downloads": baseline_checkpoint_saved_confirmation,
        "acknowledgment_sent_to_runner": baseline_checkpoint_ack_sent,
        "runner_acknowledged_before_replay": baseline_checkpoint_acknowledged,
        "browser_save_independently_verified": False,
    },
}
(run_dir / "notebook_invocation.json").write_text(json.dumps(invocation_record, indent=2) + "\n")
if invocation_error is None and baseline_checkpoint_saved_confirmation:
    packet_path = run_dir / "generation_packet.json"
    if packet_path.is_file():
        packet = json.loads(packet_path.read_text(encoding="utf-8"))
        persistence = packet.get("result_persistence")
        if isinstance(persistence, dict):
            persistence["operator_confirmed_checkpoint_visible_in_downloads"] = True
            persistence["operator_confirmation_scope"] = (
                "Operator-reported visibility in local Downloads; Colab cannot inspect or verify the browser filesystem."
            )
            packet_path.write_text(json.dumps(packet, indent=2, sort_keys=True) + "\n", encoding="utf-8")
if invocation_error is not None and not (run_dir / "failure.json").exists():
    (run_dir / "failure.json").write_text(
        json.dumps(
            {
                "status": "failed",
                "phase": "notebook_runner_or_baseline_checkpoint_handoff",
                "error_type": invocation_error["error_type"],
                "error": invocation_error["error"],
                "baseline_checkpoint_acknowledged": baseline_checkpoint_acknowledged,
                "operator_confirmed_checkpoint_visible_in_downloads": baseline_checkpoint_saved_confirmation,
                "replay_may_have_started": baseline_checkpoint_acknowledged,
                "download_acknowledgment_scope": invocation_error["download_acknowledgment_scope"],
            },
            indent=2,
        ) + "\n"
    )
# The runner writes its manifest before this wrapper adds provenance files. Refresh
# the complete hash map so the final archive covers the log, invocation, and any wrapper failure.
manifest_path = run_dir / "manifest.json"
if manifest_path.is_file():
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    artifact_sha256 = {}
    for artifact_path in sorted(run_dir.rglob("*")):
        if not artifact_path.is_file() or artifact_path.name == "manifest.json":
            continue
        digest = hashlib.sha256()
        with artifact_path.open("rb") as artifact_file:
            for chunk in iter(lambda: artifact_file.read(1024 * 1024), b""):
                digest.update(chunk)
        artifact_sha256[artifact_path.relative_to(run_dir).as_posix()] = digest.hexdigest()
    manifest["artifact_sha256"] = artifact_sha256
    temporary_manifest_path = run_dir / ".manifest.json.tmp"
    temporary_manifest_path.write_text(
        json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )
    temporary_manifest_path.replace(manifest_path)
    print("Updated manifest SHA-256 entries for", len(artifact_sha256), "artifacts")
else:
    print("Manifest hash refresh skipped: runner did not produce manifest.json")
    if invocation_error is None and runner_exit_code == 0:
        invocation_error = {
            "error_type": "FileNotFoundError",
            "error": "Runner exited successfully without producing manifest.json",
            "traceback": None,
        }
        invocation_record["status"] = "failed"
        invocation_record["error"] = invocation_error
        (run_dir / "notebook_invocation.json").write_text(
            json.dumps(invocation_record, indent=2) + "\n", encoding="utf-8"
        )
archive_path = Path(
    shutil.make_archive(
        "/content/latentgate_m1_results",
        "zip",
        root_dir=run_dir.parent,
        base_dir=run_dir.name,
    )
)
print("Created:", archive_path, "size GiB:", round(archive_path.stat().st_size / 1024**3, 3))
files.download(str(archive_path))

if invocation_error is not None:
    raise RuntimeError(
        "Runner invocation failed after available diagnostic artifacts were archived and downloaded: "
        f"{invocation_error['error_type']}: {invocation_error['error']}"
    )
if runner_exit_code != 0:
    raise subprocess.CalledProcessError(runner_exit_code, runner_command)
